# 🚀 Agoda HYBRID Crawler (direct-API + browser fallback)

Notebook gọi lại code đã test trong `agoda_direct.py` (cùng thư mục `_direct_api/`).
Chạy lần lượt: **install → import → CONFIG → RUN ALL**.

**Mục tiêu: phủ GIÁ CHÍNH XÁC NHIỀU NHẤT** (coverage không bao giờ thấp hơn notebook cũ).

Cách hoạt động của Gate 2 (crawl):
1. **Direct-API** (nhanh): warm 1 lần/KS → replay 6 tuần bằng curl_cffi.
2. **Browser fallback** (tin cậy): ô nào direct *không ra giá thật* (NA hoặc SOLD OUT nghi ngờ) → crawl lại bằng đúng phương pháp browser cũ (7 ngày/tuần + xoay fingerprint + 3 vòng retry cooldown).

→ KS dễ: direct lo, rất nhanh. KS khó (Hilton/Wyndham/Novotel…): rơi xuống browser = bằng coverage bản cũ.

Gate 0 (capture) + Gate 1 (replay) chỉ là sanity-check; Gate 2 mới là phần crawl chính.

⚠️ Phải chạy trên máy/IP của bạn (Akamai buộc cookie `_abck` theo TLS + IP).

In [1]:
# Cài 1 lần (bỏ qua nếu venv đã có sẵn)
!pip install -q curl_cffi playwright playwright-stealth pandas nest-asyncio
!playwright install chromium

source: Error encountered while sourcing file '/Users/hchinhtrung/.openclaw/completions/openclaw.fish':
source: No such file or directory

source: Error encountered while sourcing file '/Users/hchinhtrung/.openclaw/completions/openclaw.fish':
source: No such file or directory



In [2]:
import sys, os
from types import SimpleNamespace
from datetime import datetime, timedelta
import nest_asyncio; nest_asyncio.apply()

# Thư mục chứa agoda_direct.py (sửa nếu bạn đặt notebook nơi khác)
NB_DIR = '/Users/hchinhtrung/Documents/GitHub/mvillage-email-template/31.crawl-tool/_direct_api'
if NB_DIR not in sys.path:
    sys.path.insert(0, NB_DIR)
os.chdir(NB_DIR)                      # để _capture/ và DIRECT_*.csv nằm trong _direct_api/

import importlib, agoda_direct as A
importlib.reload(A)                  # nạp lại nếu bạn vừa sửa agoda_direct.py
print('✅ import agoda_direct OK | warm profiles:', [imp for _, imp in A.WARM_PROFILES])

✅ import agoda_direct OK | warm profiles: ['chrome131', 'chrome124', 'chrome120', 'chrome116']


In [3]:
# ====== ⚙️ CONFIG — sửa ở đây ======
# CAPTURE_URL: 1 URL bất kỳ copy từ cột URL của CSV (hoặc từ trình duyệt khi mở 1 KS Agoda)
CAPTURE_URL = 'https://www.agoda.com/en-gb/may-hotel-saigon_2/hotel/ho-chi-minh-city-vn.html?currencyCode=VND&los=1&adults=2&rooms=1'
ROOM        = 'Narra Double'                 # tên phòng để sanity-check giá
INPUT_CSV   = '../thy/agoda5/agoda5.csv'   # file input (đã có sẵn trong repo)
MAX_HOTELS  = 30                              # số KS crawl thử (Gate 2)
WEEKS       = 6                              # số tuần (Gate 2)

# Tuỳ chọn nâng cao (ghi đè config trong agoda_direct.py):
A.HEADLESS = True                            # đặt False để XEM trình duyệt chạy khi debug warm
# A.DAYS_PER_WEEK   = 3
# A.MAX_CONCURRENCY = 3
print('CONFIG:', CAPTURE_URL[:60], '... | room:', ROOM, '| input:', INPUT_CSV, '| max:', MAX_HOTELS, '| weeks:', WEEKS)

CONFIG: https://www.agoda.com/en-gb/may-hotel-saigon_2/hotel/ho-chi- ... | room: Narra Double | input: ../thy/agoda5/agoda5.csv | max: 30 | weeks: 6


In [ ]:
# ====== ▶️ RUN ALL — sanity (Gate 0,1) rồi crawl HYBRID (Gate 2) ======
async def run_all():
    bc = datetime.today().replace(hour=0, minute=0, second=0, microsecond=0) + timedelta(days=A.CHECKIN_OFFSET)
    print('='*64); print('🔥 GATE 0 — warm + capture (checkin =', bc.strftime('%Y-%m-%d'), ')'); print('='*64)
    cap = await A.warm_and_capture(CAPTURE_URL, bc, save=True)
    if cap.get('req') is None:
        print('\n⚠️ Gate 0 không bắt được request lúc warm — KHÔNG sao, Gate 2 vẫn chạy nhờ browser fallback.')
    elif ROOM:
        print('  🔎 Sanity giá', repr(ROOM), '->', A.extract_from_agoda(cap.get('resp_json') or {}, ROOM))

    print('\n' + '='*64); print('🧪 GATE 1 — replay verbatim + đổi ngày (sanity)'); print('='*64)
    try:
        await A.gate1_replay(SimpleNamespace(room=ROOM))
    except Exception as e:
        print('   (bỏ qua Gate 1:', e, ')')

    print('\n' + '='*64); print('🚀 GATE 2 — HYBRID crawl (direct + browser fallback)'); print('='*64)
    await A.gate2_crawl(SimpleNamespace(input=INPUT_CSV, max=MAX_HOTELS, weeks=WEEKS))

await run_all()

🔥 GATE 0 — warm + capture (checkin = 2026-07-01 )
  ✅ warm OK (profile chrome131) | rooms=True
  ✅ Bắt được room-grid (profile chrome131): POST https://www.agoda.com/api/v1/property/room-grid…
     body POST: CÓ | #cookies: 29 | response có rooms: True
     💾 Đã lưu /Users/hchinhtrung/Documents/GitHub/mvillage-email-template/31.crawl-tool/_direct_api/_capture/capture.json
  🔎 Sanity giá 'Narra Double' -> {'found': True, 'price': '2,824,074', 'room': 'Narra Double'}

🧪 GATE 1 — replay verbatim + đổi ngày (sanity)
🧪 GATE 1a — replay VERBATIM (impersonate=chrome131):
     status=200 | rooms=True
     giá 'Narra Double': {'found': True, 'price': '2,824,074', 'room': 'Narra Double'}
🧪 GATE 1b — replay ĐỔI CHECKIN 2026-07-15:
     status=200 | rooms=True

🚀 GATE 2 — HYBRID crawl (direct + browser fallback)
📂 Resume: nạp 21 dòng từ TEMP_DIRECT_agoda5.csv
🚀 HYBRID crawl | 26 KS × 6 tuần | direct trước, browser fallback | W1=2026-07-01

🏨 1/26 The Ocean Resort by Fusion Quy Nhon - Studio with P

---
### (Tuỳ chọn) Chạy RIÊNG từng gate
Bỏ dấu `#` ở dòng tương ứng rồi chạy.

In [ ]:
# await A.gate0_capture(SimpleNamespace(url=CAPTURE_URL, room=ROOM))   # chỉ Gate 0
# await A.gate1_replay(SimpleNamespace(room=ROOM))                     # chỉ Gate 1
# await A.gate2_crawl(SimpleNamespace(input=INPUT_CSV, max=MAX_HOTELS, weeks=WEEKS))   # chỉ Gate 2